In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import sqlite3
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM
from sklearn.metrics import f1_score, precision_score, recall_score

from src.config import DB_PATH, NUMERIC_METRICS, ANOMALY_LABELS
from src.detectors import load_joined_data, get_trimmed_feature_cols, OCSVM_NU, compute_cusum, CUSUM_METRICS, CUSUM_K_MULT, CUSUM_H_MULT, get_baseline_exclusions
from src.evaluate import load_ground_truth

In [2]:
df = load_joined_data(DB_PATH)
ocsvm_feature_cols = get_trimmed_feature_cols(df, NUMERIC_METRICS)

ocsvm_data = df[ocsvm_feature_cols]
ocsvm_complete_mask = ocsvm_data.notna().all(axis=1)

bad_metric_ids = get_baseline_exclusions(
    DB_PATH,
    ANOMALY_LABELS,
)

ocsvm_fit_mask = ocsvm_complete_mask & ~df["metrics_id"].isin(bad_metric_ids)
ocsvm_scored_df = df[ocsvm_complete_mask]

ocsvm_scaler = StandardScaler()
ocsvm_scaler.fit(ocsvm_data[ocsvm_fit_mask])

ocsvm_X_scaled = ocsvm_scaler.transform(ocsvm_data[ocsvm_complete_mask])

ocsvm_model = OneClassSVM(nu=OCSVM_NU)
ocsvm_model.fit(ocsvm_scaler.transform(ocsvm_data[ocsvm_fit_mask]))

anomaly_ids = load_ground_truth(DB_PATH)
ocsvm_y_true = (
    ocsvm_scored_df["metrics_id"]
    .isin(anomaly_ids)
    .astype(int)
    .to_numpy()
)

ocsvm_y_pred = (ocsvm_model.predict(ocsvm_X_scaled) == -1).astype(int)

print(
    f"n_flagged={ocsvm_y_pred.sum()}/{len(ocsvm_y_true)}  "
    f"n_true_anomalies={ocsvm_y_true.sum()}"
)


n_flagged=99/1943  n_true_anomalies=56


In [3]:
battery_check = ocsvm_scored_df.assign(is_anomaly=ocsvm_y_true)
battery_check["battery_percent"].groupby(battery_check["is_anomaly"]).describe()


,count,mean,std,min,25%,50%,75%,max
is_anomaly,,,,,,,,
0,1887.0,94.859036,13.913544,18.0,98.0,99.0,100.0,100.0
1,56.0,75.214286,13.683054,48.0,69.0,75.0,81.0,100.0


In [4]:
ablation_feature_cols = [
    c for c in ocsvm_feature_cols
    if not c.startswith("battery_percent")
]

ablation_data = df[ablation_feature_cols]
ablation_complete_mask = ablation_data.notna().all(axis=1)
ablation_fit_mask = ablation_complete_mask & ~df["metrics_id"].isin(bad_metric_ids)
ablation_scored_df = df[ablation_complete_mask]

ablation_scaler = StandardScaler()
ablation_scaler.fit(ablation_data[ablation_fit_mask])

ablation_X_scaled = ablation_scaler.transform(
    ablation_data[ablation_complete_mask]
)

ablation_model = OneClassSVM(nu=OCSVM_NU)
ablation_model.fit(
    ablation_scaler.transform(ablation_data[ablation_fit_mask])
)

ablation_y_true = (
    ablation_scored_df["metrics_id"]
    .isin(anomaly_ids)
    .astype(int)
    .to_numpy()
)

ablation_y_pred = (
    ablation_model.predict(ablation_X_scaled) == -1
).astype(int)

comparison_df = pd.DataFrame(
    [
        {
            "model": "final_ocsvm",
            "f1": f1_score(ocsvm_y_true, ocsvm_y_pred, zero_division=0),
            "precision": precision_score(
                ocsvm_y_true, ocsvm_y_pred, zero_division=0
            ),
            "recall": recall_score(
                ocsvm_y_true, ocsvm_y_pred, zero_division=0
            ),
            "flagged": int(ocsvm_y_pred.sum()),
            "scored_rows": len(ocsvm_y_true),
        },
        {
            "model": "without_battery",
            "f1": f1_score(
                ablation_y_true, ablation_y_pred, zero_division=0
            ),
            "precision": precision_score(
                ablation_y_true, ablation_y_pred, zero_division=0
            ),
            "recall": recall_score(
                ablation_y_true, ablation_y_pred, zero_division=0
            ),
            "flagged": int(ablation_y_pred.sum()),
            "scored_rows": len(ablation_y_true),
        },
    ]
)

comparison_df


,model,f1,precision,recall,flagged,scored_rows
0,final_ocsvm,0.425806,0.333333,0.589286,99,1943
1,without_battery,0.397516,0.304762,0.571429,105,1943


In [5]:
def scores_for(X):
    pred = ocsvm_model.predict(X)
    y_pred = (pred == -1).astype(int)
    return {
        "f1": f1_score(ocsvm_y_true, y_pred, zero_division=0),
        "precision": precision_score(
            ocsvm_y_true, y_pred, zero_division=0
        ),
        "recall": recall_score(
            ocsvm_y_true, y_pred, zero_division=0
        ),
    }


baseline_scores = scores_for(ocsvm_X_scaled)

rng = np.random.default_rng(42)
n_repeats = 10
rows = []

for i, col in enumerate(ocsvm_feature_cols):
    f1_drops = []
    precision_drops = []
    recall_drops = []

    for _ in range(n_repeats):
        X_perm = ocsvm_X_scaled.copy()
        X_perm[:, i] = rng.permutation(X_perm[:, i])

        scores = scores_for(X_perm)
        f1_drops.append(baseline_scores["f1"] - scores["f1"])
        precision_drops.append(
            baseline_scores["precision"] - scores["precision"]
        )
        recall_drops.append(
            baseline_scores["recall"] - scores["recall"]
        )

    rows.append(
        {
            "feature": col,
            "f1_importance": np.mean(f1_drops),
            "f1_std": np.std(f1_drops),
            "precision_importance": np.mean(precision_drops),
            "precision_std": np.std(precision_drops),
            "recall_importance": np.mean(recall_drops),
            "recall_std": np.std(recall_drops),
        }
    )

importance_df = pd.DataFrame(rows).sort_values(
    "f1_importance",
    ascending=True,
)

importance_df


,feature,f1_importance,f1_std,precision_importance,precision_std,recall_importance,recall_std
15,net_recv_mb_s_delta,0.052129,0.007049,0.060100,0.006752,-0.001786,0.005357
11,disk_usage_percent_delta,0.058104,0.002306,0.066105,0.002443,0.000000,0.000000
21,gpu_memory_percent_delta,0.070267,0.003354,0.079096,0.002699,-0.001786,0.005357
17,battery_percent_delta,0.071376,0.004454,0.076426,0.004662,0.017857,0.000000
19,gpu_percent_delta,0.073734,0.003973,0.082946,0.003012,-0.003571,0.007143
7,disk_read_mb_s_delta,0.080179,0.007384,0.086191,0.007378,0.014286,0.007143
13,net_sent_mb_s_delta,0.084006,0.008751,0.094034,0.007671,-0.008929,0.008929
1,cpu_percent_delta,0.085664,0.009715,0.094156,0.009477,0.000000,0.000000
9,disk_write_mb_s_delta,0.087533,0.009851,0.096627,0.008767,-0.003571,0.007143
23,gpu_temp_delta,0.097497,0.008927,0.106749,0.008287,-0.007143,0.008748


In [6]:
def explain_cusum_flags(db_path=DB_PATH, k_mult=CUSUM_K_MULT, h_mult=CUSUM_H_MULT):
    conn = sqlite3.connect(db_path)
    cols = ", ".join(["id AS metrics_id", "timestamp"] + CUSUM_METRICS)
    df = pd.read_sql_query(f"SELECT {cols} FROM metrics ORDER BY timestamp ASC", conn)
    conn.close()

    labeled_ids = get_baseline_exclusions(db_path, ANOMALY_LABELS)
    fit_mask = ~df["metrics_id"].isin(labeled_ids)

    per_metric_scores = {}
    per_metric_flags = {}
    for col in CUSUM_METRICS:
        baseline = df.loc[fit_mask, col].dropna()
        mu, sigma = baseline.mean(), baseline.std()
        values = df[col].ffill().bfill().to_numpy()
        score, flag = compute_cusum(values, mu, sigma, k_mult, h_mult)
        per_metric_scores[col] = score
        per_metric_flags[col] = flag

    score_df = pd.DataFrame(per_metric_scores, index=df["metrics_id"])
    flag_df = pd.DataFrame(per_metric_flags, index=df["metrics_id"])

    any_flagged = flag_df.any(axis=1)
    explanations = score_df[any_flagged].idxmax(axis=1)
    explanations = explanations.rename("triggering_metric")

    return df[any_flagged.values][["metrics_id", "timestamp"]].merge(
        explanations, left_on="metrics_id", right_index=True
    ).merge(
        score_df[any_flagged].max(axis=1).rename("cusum_score"),
        left_on="metrics_id", right_index=True
    )

explanations_df = explain_cusum_flags()
pd.set_option('display.max_rows', None)
explanations_df

,metrics_id,timestamp,triggering_metric,cusum_score
175,182,2026-09-03T07:40:20.224814+00:00,cpu_percent,1.022925
176,183,2026-09-03T07:40:52.424386+00:00,cpu_percent,1.127155
177,184,2026-09-03T07:41:25.254537+00:00,cpu_percent,1.308443
178,185,2026-09-03T07:41:57.493708+00:00,cpu_percent,1.361301
179,186,2026-09-03T07:42:29.668169+00:00,cpu_percent,1.499503
180,214,2026-09-03T08:02:16.797645+00:00,cpu_percent,1.262360
224,263,2026-09-03T08:30:16.381388+00:00,cpu_percent,1.243179
225,264,2026-09-03T08:30:48.940519+00:00,cpu_percent,1.707010
226,265,2026-09-03T08:31:20.282518+00:00,cpu_percent,2.170841
227,266,2026-09-03T08:31:51.649414+00:00,cpu_percent,2.634673


In [7]:
explanations_df["triggering_metric"].value_counts()

triggering_metric
cpu_percent    157
Name: count, dtype: int64

In [8]:
Path("data").mkdir(exist_ok=True)
importance_df.to_csv("data/ocsvm_importance.csv", index=False)